# VIN Enrichment Example

This notebook demonstrates how to enrich vehicle data using the VIN decoder API.

## Prerequisites

1. Make sure the VIN API is running:
   ```bash
   cd services/vin-api
   docker-compose up -d
   ```

2. Install required packages:
   ```bash
   pip install pandas requests
   ```

## Option 1: Using the Enrichment Script

The simplest way is to use the pre-built script:

In [ ]:
# Run the enrichment script
!python ../enrich_vehicles.py -n 100

## Option 2: Importing the Function

You can also import and use the decode function directly:

In [ ]:
import sys
import pandas as pd
sys.path.append('..')

from enrich_vehicles import decode_vin

### Decode a Single VIN

In [ ]:
# Example VIN
vin = "1HGBH41JXMN109186"

# Decode it
result = decode_vin(vin)

print(f"VIN: {vin}")
print(f"Make: {result['make']}")
print(f"Model: {result['model']}")
print(f"Year: {result['year']}")

## Option 3: Custom DataFrame Processing

For more control, process a DataFrame directly:

In [ ]:
import pandas as pd
import requests

# Load the vehicles data
df = pd.read_csv('../data/vehicles.csv', dtype=str, nrows=100)

print(f"Loaded {len(df):,} rows")
print(f"\nColumns: {', '.join(df.columns)}")

### Check Data Completeness

In [ ]:
# Check how many rows have VINs
has_vin = df['VIN'].notna() & (df['VIN'].str.len() == 17)
print(f"Rows with valid VIN: {has_vin.sum():,}")

# Check completeness
missing_year = df['year'].isna() | (df['year'] == '')
missing_make = df['manufacturer'].isna() | (df['manufacturer'] == '')
missing_model = df['model'].isna() | (df['model'] == '')

needs_enrichment = has_vin & (missing_year | missing_make | missing_model)

print(f"Rows needing enrichment: {needs_enrichment.sum():,}")
print(f"  - Missing year: {(has_vin & missing_year).sum():,}")
print(f"  - Missing make: {(has_vin & missing_make).sum():,}")
print(f"  - Missing model: {(has_vin & missing_model).sum():,}")

### View Rows That Need Enrichment

In [ ]:
# Show rows that need enrichment
df[needs_enrichment][['VIN', 'year', 'manufacturer', 'model', 'price', 'state']].head(10)

### Enrich Missing Data

In [ ]:
import time

# API endpoint
API_URL = "http://localhost:8000"

# Process rows needing enrichment
rows_to_enrich = df[needs_enrichment].copy()

print(f"Processing {len(rows_to_enrich)} rows...\n")

for idx, row in rows_to_enrich.iterrows():
    vin = row['VIN']
    
    # Decode VIN
    decoded = decode_vin(vin)
    
    if decoded:
        # Update DataFrame
        if pd.isna(row['year']) or row['year'] == '':
            df.at[idx, 'year'] = decoded['year']
        if pd.isna(row['manufacturer']) or row['manufacturer'] == '':
            df.at[idx, 'manufacturer'] = decoded['make']
        if pd.isna(row['model']) or row['model'] == '':
            df.at[idx, 'model'] = decoded['model']
        
        print(f"✓ {vin}: {decoded['year']} {decoded['make']} {decoded['model']}")
    else:
        print(f"✗ Failed to decode {vin}")
    
    # Rate limiting
    time.sleep(0.1)

print("\n✓ Enrichment complete!")

### View Enriched Data

In [ ]:
# Show the enriched rows
enriched_indices = rows_to_enrich.index
df.loc[enriched_indices, ['VIN', 'year', 'manufacturer', 'model', 'price', 'state']].head(10)

### Save Enriched Data

In [ ]:
# Save to new file
output_file = '../data/vehicles_enriched.csv'
df.to_csv(output_file, index=False)

print(f"✓ Saved enriched data to: {output_file}")

## Option 4: Batch Processing with Progress Bar

In [ ]:
# Install tqdm for progress bars
!pip install tqdm -q

In [ ]:
from tqdm import tqdm
import pandas as pd
import time

# Load data
df = pd.read_csv('../data/vehicles.csv', dtype=str, nrows=1000)

# Find rows needing enrichment
has_vin = df['VIN'].notna() & (df['VIN'].str.len() == 17)
missing_data = df['year'].isna() | df['manufacturer'].isna() | df['model'].isna()
needs_enrichment = has_vin & missing_data

rows_to_process = df[needs_enrichment].copy()

print(f"Processing {len(rows_to_process):,} VINs...\n")

# Process with progress bar
successful = 0
failed = 0

for idx, row in tqdm(rows_to_process.iterrows(), total=len(rows_to_process), desc="Decoding VINs"):
    vin = row['VIN']
    decoded = decode_vin(vin)
    
    if decoded:
        if pd.isna(row['year']):
            df.at[idx, 'year'] = decoded['year']
        if pd.isna(row['manufacturer']):
            df.at[idx, 'manufacturer'] = decoded['make']
        if pd.isna(row['model']):
            df.at[idx, 'model'] = decoded['model']
        successful += 1
    else:
        failed += 1
    
    time.sleep(0.1)

print(f"\n✓ Complete! {successful} successful, {failed} failed")

## Data Analysis

Once enriched, you can analyze the data:

In [ ]:
# Load enriched data
df = pd.read_csv('../data/vehicles_enriched.csv', dtype=str)

# Convert year to numeric
df['year_num'] = pd.to_numeric(df['year'], errors='coerce')

# Top manufacturers
print("Top 10 Manufacturers:")
print(df['manufacturer'].value_counts().head(10))

print("\nYear Distribution:")
print(df['year_num'].describe())

In [ ]:
# Visualize
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Top manufacturers
df['manufacturer'].value_counts().head(10).plot(kind='barh', ax=ax1)
ax1.set_title('Top 10 Manufacturers')
ax1.set_xlabel('Count')

# Year distribution
df['year_num'].hist(bins=30, ax=ax2)
ax2.set_title('Vehicle Year Distribution')
ax2.set_xlabel('Year')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.show()